# paper-translator: Colab サーバー

GPU ランタイムで実行する。VS Code の Colab 拡張機能でカーネルに Colab（L4 / A100）を選んでも、ブラウザの Colab でも動く。
カーネルは Colab のマシン上で動くため、コードは GitHub から取得する（手元の変更は push してから下の取得セルを再実行）。

API を起動して cloudflared で公開し、表示された URL をフロントの「API URL」欄に入れる。

**GPU の選び方**: モデルカードは BF16 対応 GPU（L4 / A100 など）を前提にしている。T4 は BF16 非対応で、fp16 で動くかは未確認。

| モデル | BF16 の重みサイズ |
|---|---|
| `pfnet/plamo-3-translate-2b`（既定） | 約 5.8 GB |
| `pfnet/plamo-3-translate-8b` | 約 17.1 GB |


In [ ]:
!nvidia-smi

In [ ]:
import os

REPO = "/content/paper-translator"
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/Kame09-M/paper-translator.git {REPO}
%cd {REPO}/backend
!git log --oneline -1
!pip install -q -U -r requirements-gpu.txt

# 取り込んだコードの変更を、カーネル再起動なしで反映する
%load_ext autoreload
%autoreload 2

## 設定

Gemini の API キーは Colab の「シークレット」に `GEMINI_API_KEY` として登録する（ノートブックに直接書かない）。未登録なら用語説明と要約はダミーのまま動く。


In [ ]:
import os

os.environ["MOCK"] = "0"  # 翻訳を本物にする
os.environ["PLAMO_MODEL"] = "pfnet/plamo-3-translate-2b"

try:
    from google.colab import userdata

    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
except Exception:
    pass  # VS Code の Colab 拡張機能でシークレットが使えるかは未確認
if not os.environ.get("GEMINI_API_KEY"):
    print("GEMINI_API_KEY 未設定: 用語説明と要約はダミー")

## 翻訳だけ先に試す

モデルの読み込み（初回はダウンロードあり）と、用語集・履歴付きの翻訳を確かめる。


In [ ]:
from app.schemas import GlossaryEntry
from app.services import translate

glossary = [GlossaryEntry(term="attention", translation="アテンション", paragraph_ids=[0, 1])]
paras = [
    "The Transformer relies entirely on attention to draw global dependencies.",
    "Self-attention relates different positions of a single sequence.",
]
for src, tgt in zip(paras, translate.translate_paragraphs(paras, "ja", glossary)):
    print(src, "\n→", tgt, "\n")

## 実物の論文で確かめる（時間・用語集の効果）

公開論文「Attention Is All You Need」の先頭 N 段落を訳し、1段落あたりの時間を測る。
用語集あり／なしの訳も並べるので、訳語の揺れを目で見比べる（簡易評価）。


In [ ]:
import time
import urllib.request

from app.services import pdf, terms

urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", "/content/attention.pdf")
paragraphs = pdf.extract_paragraphs(open("/content/attention.pdf", "rb").read())
print("段落数:", len(paragraphs))

t = time.time()
glossary = terms.build_glossary(paragraphs, "ja")
print(f"用語集 {len(glossary)} 語: {time.time() - t:.1f} 秒")
for g in glossary:
    print(f"  {g.term} → {g.translation}")

In [ ]:
N = 5
sample = paragraphs[4 : 4 + N]  # 冒頭の著者・注記を飛ばす

t = time.time()
with_glossary = translate.translate_paragraphs(sample, "ja", glossary)
sec = (time.time() - t) / N
print(f"用語集あり: 1段落あたり {sec:.1f} 秒 → 全 {len(paragraphs)} 段落なら約 {sec * len(paragraphs) / 60:.1f} 分")

without_glossary = translate.translate_paragraphs(sample, "ja", [])

for src, a, b in zip(sample, with_glossary, without_glossary):
    print("原文:", src[:200], "\n用語集あり:", a, "\n用語集なし:", b, "\n")

## API サーバー起動


In [ ]:
import subprocess

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
)
!sleep 5 && curl -s localhost:8000/api/health

## cloudflared で公開

出力に出る `https://*.trycloudflare.com` の URL をフロントに入れる。このセルは動かし続ける。


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8000